# Dimensionality Reduction: From Theory to Practice

This notebook explores how to manage high-dimensional data, covering:
1. **The Curse of Dimensionality:** Proving mathematically why distance metrics fail in high-D space.
2. **Principal Component Analysis (PCA):** Extracting linear variance.
3. **Singular Value Decomposition (SVD):** Matrix factorization for sparse data.
4. **t-SNE vs. PCA:** Visualizing non-linear high-dimensional clusters.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.datasets import make_classification

sns.set_theme(style="whitegrid")

### 1. The Curse of Dimensionality (Distance Sparsity)
In high-dimensional space, data becomes extremely sparse. Let's prove this by generating random points in different dimensions (2D, 10D, 100D, 1000D) and plotting the distribution of the distances between them.

**Hypothesis:** As dimensions increase, the distance between any two points converges. "Nearest neighbors" loses its meaning because everything is effectively the exact same distance apart.

In [ ]:
dims = [2, 10, 100, 1000]
plt.figure(figsize=(10, 6))

np.random.seed(42)
for d in dims:
    # Generate 200 random points in d-dimensional space
    points = np.random.uniform(0, 1, (200, d))
    # Calculate all pairwise distances between the points
    distances = pdist(points)
    # Plot the density of these distances
    sns.kdeplot(distances, label=f'{d} Dimensions', fill=True, alpha=0.3)

plt.title("The Curse of Dimensionality: Distance Distribution", fontsize=14)
plt.xlabel("Pairwise Distance Between Points")
plt.ylabel("Density")
plt.legend()
plt.show()

print("--- INSIGHT ---")
print("Look at 2D (Blue): Distances are spread out. Some points are close (distance ~0.2), some are far (distance ~1.2).")
print("Look at 1000D (Red): The curve is a massive, tight spike. Every point is exactly ~13 units away from every other point. Distance metrics (KNN, K-Means) completely break down here!")

### 2. Principal Component Analysis (PCA)
PCA is a linear technique that transforms original features into a smaller set of uncorrelated variables. Let's create a 3-dimensional dataset that actually lies mostly flat on a 2D plane, and use PCA to compress it.

In [ ]:
# Generate 3D data where the 3rd dimension (Z) is just a linear combination of X and Y
np.random.seed(42)
x = np.random.normal(0, 1, 300)
y = np.random.normal(0, 1, 300)
z = (2 * x) + (0.5 * y) + np.random.normal(0, 0.1, 300) # Heavy correlation + slight noise

X_3d = np.column_stack((x, y, z))

# Apply PCA to reduce from 3D down to 2D
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_3d)

print(f"Original Data Shape: {X_3d.shape}")
print(f"Reduced Data Shape: {X_2d.shape}")
print(f"Variance Captured by 2 Components: {np.sum(pca.explained_variance_ratio_) * 100:.2f}%\n")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(X_3d[:, 0], X_3d[:, 1], alpha=0.6, color='teal')
axes[0].set_title("Original Data (Viewing X & Y axes)")

axes[1].scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.6, color='purple')
axes[1].set_title("PCA Compressed Data (Principal Components 1 & 2)")

plt.show()
print("Notice how PCA rotated and scaled the axes to capture maximum variance. We dropped an entire dimension but lost less than 1% of the mathematical information!")

### 3. Singular Value Decomposition (SVD)
While PCA centers data (subtracts the mean), Truncated SVD does not. This makes it ideal for reducing **sparse matrices**, like TF-IDF text vectors or User-Item recommendation matrices, where subtracting the mean would destroy the sparsity and crash your RAM.

In [ ]:
from scipy.sparse import csr_matrix

# Imagine a massive dataset of 1000 users and 5000 movies.
# Most users have only rated 1 or 2 movies (highly sparse).
sparse_data = csr_matrix(np.random.binomial(1, 0.05, size=(1000, 5000)))

print(f"Original Sparse Matrix Shape: {sparse_data.shape}")

# Apply SVD to extract latent factors (e.g., hidden "genres" of movies)
svd = TruncatedSVD(n_components=10)
latent_factors = svd.fit_transform(sparse_data)

print(f"Reduced Latent Factor Shape: {latent_factors.shape}")
print("Result: We reduced 5000 individual movies down to 10 underlying 'latent themes'.")

### 4. t-SNE vs PCA for High-Dimensional Visualization
PCA relies on straight lines (linear). High-dimensional data often has complex, curved, non-linear manifolds. **t-SNE** is designed specifically to map these complex 50D or 100D relationships down into beautifully separated 2D visual clusters.

In [ ]:
# Generate a highly complex 50-Dimensional dataset with 3 underlying clusters
X_high, y_labels = make_classification(
    n_samples=600, 
    n_features=50,       # 50 dimensions!
    n_informative=15, 
    n_redundant=10, 
    n_clusters_per_class=1, 
    n_classes=3, 
    random_state=42
)

# 1. Linear Projection (PCA)
pca_proj = PCA(n_components=2).fit_transform(X_high)

# 2. Non-Linear Embedding (t-SNE)
# Perplexity balances attention between local and global aspects of your data.
tsne_proj = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_high)

# Visual Showdown
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(x=pca_proj[:, 0], y=pca_proj[:, 1], hue=y_labels, palette='Set2', ax=axes[0], s=60)
axes[0].set_title("Linear: PCA Projection (Fails to separate)", fontsize=14)

sns.scatterplot(x=tsne_proj[:, 0], y=tsne_proj[:, 1], hue=y_labels, palette='Set2', ax=axes[1], s=60)
axes[1].set_title("Non-Linear: t-SNE Projection (Beautiful separation)", fontsize=14)

plt.tight_layout()
plt.show()

print("--- THE FINAL TAKEAWAY ---")
print("Because the 50D data structure was non-linear, PCA mashed the classes together. t-SNE successfully preserved the 'local neighborhoods' of the high-dimensional points, making it the ultimate tool for 2D human visualization of complex data.")